# Patient Recommender System

Using RAG framework to retrieve cases and provide summaries for patients that are of similar nature

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 30/03/2026   | Martin | CREATE  | Notebook created. Started working on document format for patient profile | 
| 31/03/2026   | Martin | UPDATE  | Format document for both MIMIC and Synthea data | 
| 01/04/2026   | Martin | UPDATE  | Document creation for MIMIC data |
| 02/04/2026   | Martin | UPDATE  | Document creation for Synthea |
| 08/04/2026   | Martin | UPDATE  | Document generation function for both MIMIC and Synthea documents |
| 15/04/2026   | Martin | UPDATE  | Notebook cleanup |

# Content

* [Patient Snapshot](#patient-snapshot)
  - [MIMIC Data](#mimic-data)
  - [Synthea Data](#synthea-data)
* [Idea for Document](#idea-for-document)
* [Testing Document Creation](#testing-document-creation)

# Patient Snapshot

Explore the availability of data from both Synthea and MIMIC resources to craft the patient's profile

In [1]:
import duckdb

In [2]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")

In [3]:
con.sql("SHOW TABLES FROM silver;")

┌───────────────────────────────────┐
│               name                │
│              varchar              │
├───────────────────────────────────┤
│ condition                         │
│ encounter                         │
│ location                          │
│ medication_dispense               │
│ obs_vitals                        │
│ organization                      │
│ patient                           │
│ procedure                         │
│ synthea_allergy_intolerance       │
│ synthea_careplan                  │
│ synthea_careteam                  │
│ synthea_condition                 │
│ synthea_encounter                 │
│ synthea_immunization              │
│ synthea_medication                │
│ synthea_medication_administration │
│ synthea_medication_request        │
│ synthea_observation               │
│ synthea_patient                   │
│ synthea_procedure                 │
├───────────────────────────────────┤
│              20 rows              │
└───────────

In [ ]:
con.close()

## MIMIC Data

Observations from patients:

1. f040bdf0-6951-5194-a400-fac7a247583c (15174162)
2. f7ed2ddf-9129-51d0-9bad-0d53665559db (15653234)
3. d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03 (15653252)

<u>Data Present</u>

| Column | Table | Type |  
| ------ | ----- | ---- |
| name | patient | categorical |
| gender | patient | categorical |
| birth_date | patient | date (YYYY-DD-MM) |
| race | patient | categorical |
| ethnicity | patient | categorical |
| marital_status | patient | categorical |

In [ ]:
# Define different id as examples
id1 = "f040bdf0-6951-5194-a400-fac7a247583c"
id2 = "f7ed2ddf-9129-51d0-9bad-0d53665559db"
id3 = "d4566ed7-5ee5-5cb1-bb88-b08f5b0a8c03"

In [14]:
con.sql(f"""
SELECT 
  c.icd_code,
  c.icd_name,
  e.start_timestamp,
  e.end_timestamp,
  e.admit_source,
  e.discharge_disposition
FROM silver.condition c
LEFT JOIN silver.encounter e ON c.encounter_id = e.resource_id
WHERE c.patient_id='{id1}';
""")

┌──────────┬─────────────────────────────────────────────────────────────────────────────────┬─────────────────────┬─────────────────────┬──────────────┬───────────────────────┐
│ icd_code │                                    icd_name                                     │   start_timestamp   │    end_timestamp    │ admit_source │ discharge_disposition │
│ varchar  │                                     varchar                                     │      timestamp      │      timestamp      │   varchar    │        varchar        │
├──────────┼─────────────────────────────────────────────────────────────────────────────────┼─────────────────────┼─────────────────────┼──────────────┼───────────────────────┤
│ R502     │ Drug induced fever                                                              │ 2136-04-19 02:12:00 │ 2136-04-19 10:10:00 │ WALK IN      │ HOME                  │
│ R0902    │ Hypoxemia                                                                       │ 2136-07-07 12:1

In [36]:
con.sql(f"SELECT * FROM silver.condition WHERE patient_id='{id1}';")

┌──────────────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┐
│             resource_id              │ icd_code │                                    icd_name                                     │              patient_id              │ condition_category  │             encounter_id             │
│               varchar                │ varchar  │                                     varchar                                     │               varchar                │       varchar       │               varchar                │
├──────────────────────────────────────┼──────────┼─────────────────────────────────────────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼──────────────────────────────────────┤
│ ff4a1fc3-16ff-5f20-ac65-2f08531c7fd0 │ R0902    │ Hypoxemia   

In [45]:
con.sql(f"SELECT * FROM silver.obs_vitals WHERE patient_id='{id1}';")

┌──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬────────┬────────────────┐
│             resource_id              │              patient_id              │             encounter_id             │ effective_datetime  │ loinc_code │ value  │      unit      │
│               varchar                │               varchar                │               varchar                │      timestamp      │  varchar   │ double │    varchar     │
├──────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼────────┼────────────────┤
│ 31cf70a6-5495-56b9-ac34-f4c193732530 │ f040bdf0-6951-5194-a400-fac7a247583c │ ef4161fd-b864-5f39-a1c9-aea9ba376086 │ 2136-07-07 12:14:00 │ 8867-4     │  145.0 │ beats/minute   │
│ 2ee943a9-29af-580f-b8d8-7007775c1882 │ f040bdf0-6951-5194-a400-fac7a247583c │ ef4161fd-b864-5f39-a

### Transferring MIMIC silver to gold

Here we transform silver MIMIC data to gold tables

In [ ]:
# Want to copy the schema of the silver.patient table to first create
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.patient (
  resource_id VARCHAR,
  name VARCHAR,
  gender VARCHAR,
  birth_date VARCHAR,
  race VARCHAR,
  ethnicity VARCHAR,
  marital_status VARCHAR,
  created_at TIMESTAMP DEFAULT now(),
)
""")

# Create the conditions table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.conditions (
  patient_id VARCHAR,
  icd_code VARCHAR,
  icd_name VARCHAR,
  start_timestamp TIMESTAMP,
  end_timestamp TIMESTAMP
)
""")

# Create the procedures table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.procedures (
  patient_id VARCHAR,
  performed_datetime TIMESTAMP,
  snomed_ct_id VARCHAR,
  snomed_ct_procedure VARCHAR
)
""")

# Create the Medication Dispense table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.medication_dispense (
  patient_id VARCHAR,
  administered_date TIMESTAMP,
  medication_code VARCHAR,
  medication VARCHAR
)
""")

# Create the Vitals table
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.vitals (
  patient_id VARCHAR,
  observation_date TIMESTAMP,
  obs_code VARCHAR,
  value DOUBLE,
  unit VARCHAR
)
""")

In [205]:
con.sql("SELECT * FROM silver.patient")

┌──────────────────────────────────────┬──────────────────┬─────────┬────────────┬───────────────────────────┬────────────────────────┬─────────────┬──────────┬────────────────┬──────────────────────────────────────┐
│             resource_id              │       name       │ gender  │ birth_date │           race            │       ethnicity        │ patient_num │ language │ marital_status │           organization_id            │
│               varchar                │     varchar      │ varchar │  varchar   │          varchar          │        varchar         │   varchar   │ varchar  │    varchar     │               varchar                │
├──────────────────────────────────────┼──────────────────┼─────────┼────────────┼───────────────────────────┼────────────────────────┼─────────────┼──────────┼────────────────┼──────────────────────────────────────┤
│ 7d101f5f-aba3-586c-94b2-6055c90992a2 │ Patient_15173979 │ male    │ 2072-02-02 │ unknown                   │ NULL                 

In [208]:
# On initial insert will take all patients
con.execute("""
INSERT INTO mimic_fhir.temp.patient
SELECT
  resource_id,
  name,
  gender,
  birth_date,
  race,
  ethnicity,
  marital_status,
  now()
FROM silver.patient
""")

In [46]:
# On subsequent it inserts it will compare the current patient id with the silver tables ones - This should be a pipeline script
con.execute("""
INSERT INTO mimic_fhir.temp.patient
SELECT
  t1.*,
  0 AS documented
FROM silver.patient t1
WHERE NOT EXISTS (
  SELECT 1 FROM mimic_fhir.temp.patient t2 WHERE t1.resource_id = t2.resource_id
);
""")

In [ ]:
# Conditions
con.sql("""
INSERT INTO mimic_fhir.temp.conditions
SELECT
  c.patient_id,
  c.icd_code,
  c.icd_name,
  e.start_timestamp,
  e.end_timestamp
FROM silver.condition c
LEFT JOIN silver.encounter e ON c.encounter_id = e.resource_id;
""")

# Procedure
con.sql("""
INSERT INTO mimic_fhir.temp.procedures
SELECT 
  patient_id,
  performed_datetime,
  snomed_ct_id,
  snomed_ct_procedure
FROM silver.procedure;
""")

# Medication Dispense
con.sql("""
INSERT INTO mimic_fhir.temp.medication_dispense        
SELECT
  patient_id,
  handed_over_timestamp,
  medication_code,
  medication_name
FROM silver.medication_dispense
""")

# Vitals
con.sql("""
INSERT INTO mimic_fhir.temp.vitals
SELECT
  patient_id,
  effective_datetime,
  loinc_code,
  value,
  unit
FROM silver.obs_vital
""")

In [16]:
con.close()

## Synthea Data

Observations from patients:

1. 24283ca3-2f37-a4c1-e740-97f060a230a6
2. 2e312e34-cc56-6a67-6642-c261c2f4838f
3. a3d61946-69d4-cd45-0792-f8b941bd6f73

<u>Data Present</u>

| Column | Table | Type | Notes | 
| ------ | ----- | ---- | ----- |
| name | patient | categorical | Combine first_name + middle_name + last_name |
| gender | patient | categorical | |
| birth_date | patient | date (YYYY-DD-MM) | |
| race | patient | categorical | |
| ethnicity | patient | categorical | |
| marital_status | patient | categorical | Convert to MIMIC format |


In [ ]:
# Define sample synthea ids to check data integrity
syn_id1 = "24283ca3-2f37-a4c1-e740-97f060a230a6"
syn_id2 = "2e312e34-cc56-6a67-6642-c261c2f4838f"
syn_id3 = "a3d61946-69d4-cd45-0792-f8b941bd6f73"

In [13]:
con.sql("SELECT * FROM silver.synthea_patient;")

┌──────────────────────────────────────┬─────────────┬─────────────┬────────────┬─────────┬───────────────────────────────────────────┬────────────────────────┬────────────┬──────────────┬────────────────┬──────────────────┬─────────┬─────────┬───────────────────────────────────────┬────────────────────┬────────────────────┬─────────────────────────────────┬─────────────────────────────┐
│             resource_id              │ first_name  │ middle_name │ last_name  │ gender  │                   race                    │       ethnicity        │ birth_date │ phone_number │ marital_status │       city       │  state  │ country │            street_address             │      latitude      │     longitude      │ disablility_adjusted_life_years │ quality_adjusted_life_years │
│               varchar                │   varchar   │   varchar   │  varchar   │ varchar │                  varchar                  │        varchar         │  varchar   │   varchar    │    varchar     │     varchar 

In [44]:
con.sql(f"""
SELECT 
  c.condition_code,
  c.condition,
  c.condition_type,
  c.patient_id,
  c.onset_timestamp,
  c.recorded_timestamp,
  e.procedure_code,
  e.procedure_name,
  m.medication_code,
  m.medication,
  m.effective_timestamp,
  m.reason_id,
  m.reason,
  m.reason_type,
FROM silver.synthea_condition c
LEFT JOIN silver.synthea_encounter e ON c.encounter_id = e.resource_id
LEFT JOIN silver.synthea_medication_administration m ON c.encounter_id = m.context_id
WHERE c.patient_id='{syn_id2}';
""")

┌────────────────┬───────────────────────────────────────┬────────────────┬──────────────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬─────────────────────────────────┬─────────────────┬───────────────────────────────────────┬─────────────────────┬──────────────────────────────────────┬───────────────────────────────────┬─────────────┐
│ condition_code │               condition               │ condition_type │              patient_id              │   onset_timestamp   │ recorded_timestamp  │ procedure_code │         procedure_name          │ medication_code │              medication               │ effective_timestamp │              reason_id               │              reason               │ reason_type │
│    varchar     │                varchar                │    varchar     │               varchar                │      timestamp      │      timestamp      │    varchar     │             varchar             │     varchar     │               

### Moving from silver to gold

Moving the Synthea data from silver tables to useable gold marts

In [ ]:
# Table creation
con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.synthea_patient (
  resource_id VARCHAR,
  first_name VARCHAR,
  middle_name VARCHAR,
  last_name VARCHAR,
  gender VARCHAR,
  race VARCHAR,
  ethnicity VARCHAR,
  birth_date VARCHAR,
  marital_status VARCHAR,
  documented INTEGER DEFAULT 0
)
""")

con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.synthea_condition (
  patient_id VARCHAR,
  onset_timestamp TIMESTAMP,
  diagnosis_timestamp TIMESTAMP,
  condition_code VARCHAR,
  condition VARCHAR
)
""")

con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.synthea_procedure (
  patient_id VARCHAR,
  performed_date TIMESTAMP,
  procedure_code VARCHAR,
  procedure VARCHAR
)
""")

con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.synthea_medication_dispense (
  patient_id VARCHAR,
  administered_date TIMESTAMP,
  medication_code VARCHAR,
  medication VARCHAR
)
""")

con.execute("""
CREATE TABLE IF NOT EXISTS mimic_fhir.temp.synthea_vitals (
  patient_id VARCHAR,
  effective_timestamp TIMESTAMP,
  vitals_code VARCHAR,
  value DOUBLE,
  unit VARCHAR
)
""")

In [174]:
con.sql("""
INSERT INTO mimic_fhir.temp.synthea_patient
SELECT
  resource_id,
  first_name,
  middle_name,
  last_name,
  gender,
  race,
  ethnicity,
  birth_date,
  marital_status,
  0
FROM silver.synthea_patient;
""")

con.sql("""
INSERT INTO mimic_fhir.temp.synthea_condition
SELECT
  c.patient_id,
  c.onset_timestamp,
  e.end_date,
  c.condition_code,
  c.condition
FROM silver.synthea_condition c
LEFT JOIN silver.synthea_encounter e ON c.encounter_id = e.resource_id
WHERE c.condition_type = 'disorder';
""")

con.sql("""
INSERT INTO mimic_fhir.temp.synthea_procedure
SELECT
  patient_id,
  start_timestamp,
  procedure_code,
  reason
FROM silver.synthea_procedure;
""")

con.sql("""
INSERT INTO mimic_fhir.temp.synthea_medication_dispense
SELECT
  patient_id,
  effective_timestamp,
  medication_code,
  medication
FROM silver.synthea_medication_administration
""")

con.sql("""
INSERT INTO mimic_fhir.temp.synthea_vitals
SELECT
  patient_id,
  issued_timestamp,
  measure_code,
  value,
  unit
FROM silver.synthea_observation
WHERE record = 'vital-signs';
""")

# Idea for Document

Patient Information
- Name: {name}
- Gender: {gender}
- Birth Date: {birth_date}
- Race: {race}
- Ethnicity: {ethnicity}
- Marital Status: {marital_status}

Conditions
| Onset Date | Diagnosis Date | ICD-10 Code | Condition |
| ---------- | -------------- | ----------- | --------- |
| {start_date} | {end_date} | {icd_10_code} | {condition_name} 
...

Procedures
| Performed Date |  snomed_ct Code | Procedure |
| -------------- |  ----------- | --------- |
| {performed_date} | {procedure_code} | {procedure} |
...

Medication Dispensed
| Administered Date | Medication Code | Medication |
| ----------------- | --------------- | ---------- |
| {effective_timestamp} | {medication_code} | {medication} |
...

Vitals (Observation)
| Observation Date | Measured Vitals |
| ---------------- | --------------- |
| {effective_timestamp} | {vitals_name}: {value}{unit}, {vitals_name}: {value}{unit} |
...



In [9]:
con.close()

# Testing Document Creation

Using the document creation functions for both MIMIC and Synthea, we generate documents for all available patients

In [78]:
import duckdb
import numpy as np
from langchain_core.documents import Document
from datetime import datetime
from typing import Literal

In [24]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")

In [26]:
con.sql("SHOW TABLES FROM recommend")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ mi_condition            │
│ mi_medication_dispense  │
│ mi_patient              │
│ mi_procedure            │
│ mi_vitals               │
│ syn_condition           │
│ syn_medication_dispense │
│ syn_patient             │
│ syn_procedure           │
│ syn_vitals              │
├─────────────────────────┤
│         10 rows         │
└─────────────────────────┘

## Document generation

In [81]:
def get_relevant_ids(table1: str = "mi_patient", table2: str = "mi_condition") -> np.ndarray:
  return con.sql(
  f"""
SELECT resource_id FROM recommend.{table1}
INTERSECT
SELECT patient_id FROM recommend.{table2};
  """
  ).fetchnumpy()['resource_id']

In [82]:
def create_mimic_patient_document(
  patient_dict: dict,
  conditions_dict: dict,
  procedures_dict: dict,
  medication_dict: dict,
  vitals_dict: dict
) -> Document:
  # Initial patient info
  page_content = f"""
Patient Information:
- Name: {patient_dict.get('name', '')}
- Gender: {patient_dict.get('gender', '')}
- DOB: {patient_dict.get('birth_date', '')}
- Race: {patient_dict.get('race', '')}
- Ethnicity: {patient_dict.get('ethnicity', '')}
- Marital Status: {patient_dict.get('marital_status', '')}
  """

  # Conditions info
  base_conditions = """
Conditions
ONSET DATE, DIAGNOSIS DATE, CODE, CONDITION
"""
  for i in range(len(conditions_dict['patient_id'])):
    start_time = conditions_dict['start_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    end_time = conditions_dict['end_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_conditions += f"{i+1}. {start_time}, {end_time}, {conditions_dict['icd_code'][i]}, {conditions_dict['icd_name'][i]}\n"
  page_content += f"\n{base_conditions}"

  # Procedures info
  base_procedures = """
Procedures
PERFORMED DATE, CODE, PROCEDURE
"""
  for i in range(len(procedures_dict['patient_id'])):
    procedure_time = procedures_dict['performed_datetime'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_procedures += f"{i+1}. {procedure_time}, {procedures_dict['snomed_ct_id'][i]}, {procedures_dict['snomed_ct_procedure'][i]}\n"
  page_content += f"\n{base_procedures}"

  # Medication Dispense
  base_medications = """
Medication Dispensed
ADMINISTERED DATE, CODE, MEDICATION
"""
  for i in range(len(medication_dict['patient_id'])):
    administered_date = medication_dict['administered_date'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_medications += f"{i+1}. {administered_date}, {medication_dict['medication_code'][i]}, {medication_dict['medication'][i]}\n"
  page_content += f"\n{base_medications}"

  # Vitals
  base_vitals = """
Vitals
OBSERVATION DATE, VITALS
  """
  obs_vitals = ""
  curr_date = ""
  for i in range(len(vitals_dict['patient_id'])):
    observation_date = vitals_dict['observation_date'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    if curr_date == observation_date:
      obs_vitals += f" | {vitals_dict['obs_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]} "
      if i == len(vitals_dict['patient_id'])-1:
        base_vitals += obs_vitals
    else:
      curr_date = observation_date
      base_vitals += obs_vitals
      if i == 0:
        obs_vitals = f"{observation_date}, {vitals_dict['obs_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]}"
      else:
        obs_vitals = f"\n{observation_date}, {vitals_dict['obs_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]}"
  page_content += f"\n{base_vitals}"

  return page_content

In [83]:
def create_synthea_patient_document(
  patient_dict: dict,
  conditions_dict: dict,
  procedures_dict: dict,
  medication_dict: dict,
  vitals_dict: dict
) -> Document:
  # Initial patient info
  page_content = f"""
Patient Information:
- Name: {patient_dict.get('name', '')}
- Gender: {patient_dict.get('gender', '')}
- DOB: {patient_dict.get('birth_date', '')}
- Race: {patient_dict.get('race', '')}
- Ethnicity: {patient_dict.get('ethnicity', '')}
- Marital Status: {patient_dict.get('marital_status', '')}
  """

  # Conditions info
  base_conditions = """
Conditions
ONSET DATE, DIAGNOSIS DATE, CODE, CONDITION
"""
  for i in range(len(conditions_dict['patient_id'])):
    start_time = conditions_dict['onset_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    end_time = conditions_dict['diagnosis_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_conditions += f"{i+1}. {start_time}, {end_time}, {conditions_dict['condition_code'][i]}, {conditions_dict['condition'][i]}\n"
  page_content += f"\n{base_conditions}"

  # Procedures info
  base_procedures = """
Procedures
PERFORMED DATE, CODE, PROCEDURE
"""
  for i in range(len(procedures_dict['patient_id'])):
    procedure_time = procedures_dict['performed_date'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_procedures += f"{i+1}. {procedure_time}, {procedures_dict['procedure_code'][i]}, {procedures_dict['procedure'][i]}\n"
  page_content += f"\n{base_procedures}"

  # Medication Dispense
  base_medications = """
Medication Dispensed
ADMINISTERED DATE, CODE, MEDICATION
"""
  for i in range(len(medication_dict['patient_id'])):
    administered_date = medication_dict['administered_date'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    base_medications += f"{i+1}. {administered_date}, {medication_dict['medication_code'][i]}, {medication_dict['medication'][i]}\n"
  page_content += f"\n{base_medications}"

  # Vitals
  base_vitals = """
Vitals
OBSERVATION DATE, VITALS
  """
  obs_vitals = ""
  curr_date = ""
  for i in range(len(vitals_dict['patient_id'])):
    observation_date = vitals_dict['effective_timestamp'][i].astype('datetime64[s]').item().strftime('%m-%d-%Y %H:%M:%S')
    if curr_date == observation_date:
      obs_vitals += f" | {vitals_dict['vitals_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]} "
      if i == len(vitals_dict['patient_id'])-1:
        base_vitals += obs_vitals
    else:
      curr_date = observation_date
      base_vitals += obs_vitals
      if i == 0:
        obs_vitals = f"{observation_date}, {vitals_dict['vitals_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]}"
      else:
        obs_vitals = f"\n{observation_date}, {vitals_dict['vitals_code'][i]}: {vitals_dict['value'][i]}{vitals_dict['unit'][i]}"
  page_content += f"\n{base_vitals}"

  return page_content

In [ ]:
def generate_docs(ids: np.ndarray, doc_type: Literal["mimic", "synthea"]):
  prefix = "mi_" if doc_type == "mimic" else "syn_"
  for id in ids:
    # ---- Create necessary dictionaries for population --------------------
    patient = con.sql(f"SELECT * FROM recommend.{prefix}patient WHERE resource_id='{id}';").fetchnumpy()
    patient = {k: v[0] for k, v in patient.items()}
    condition = con.sql(f"SELECT * FROM recommend.{prefix}condition WHERE patient_id='{id}';").fetchnumpy()
    procedure = con.sql(f"SELECT * FROM recommend.{prefix}procedure WHERE patient_id = '{id}';").fetchnumpy()
    medication = con.sql(f"SELECT * FROM recommend.{prefix}medication_dispense WHERE patient_id = '{id}';").fetchnumpy()
    vitals = con.sql(
      f"SELECT * FROM recommend.{prefix}vitals WHERE patient_id = '{id}'"
      f"{' AND obs_code IS NOT NULL' if doc_type == 'mimic' else ''} "
      f"ORDER BY {'observation_date' if doc_type == 'mimic' else 'effective_timestamp'};"
    ).fetchnumpy()

    if doc_type == "mimic":
      # ---- Create patient docs --------------------
      doc = create_mimic_patient_document(
        patient,
        condition,
        procedure,
        medication,
        vitals
      )
    else:
      doc = create_synthea_patient_document(
        patient,
        condition,
        procedure,
        medication,
        vitals
      )
    print(doc)
    break


In [102]:
ids = get_relevant_ids("mi_patient", "mi_condition")
generate_docs(ids, doc_type="mimic")


Patient Information:
- Name: Patient_15174865
- Gender: male
- DOB: 2134-03-01
- Race: White
- Ethnicity: Not Hispanic or Latino
- Marital Status: M
  

Conditions
ONSET DATE, DIAGNOSIS DATE, CODE, CONDITION
1. 03-01-2183 19:01:00, 03-02-2183 01:45:00, R42, Dizziness and giddiness
2. 08-13-2184 17:22:00, 08-13-2184 23:59:00, I82432, Acute embolism and thrombosis of left popliteal vein


Procedures
PERFORMED DATE, CODE, PROCEDURE
1. 03-02-2183 01:31:00, 410188000, Taking patient vital signs assessment
2. 03-01-2183 19:01:00, 410188000, Taking patient vital signs assessment
3. 03-01-2183 21:49:00, 410188000, Taking patient vital signs assessment
4. 08-13-2184 17:22:00, 386478007, Triage: emergency center
5. 03-01-2183 23:35:00, 410188000, Taking patient vital signs assessment
6. 03-01-2183 19:01:00, 386478007, Triage: emergency center
7. 08-13-2184 21:34:00, 410188000, Taking patient vital signs assessment
8. 03-01-2183 21:50:00, 410188000, Taking patient vital signs assessment
9. 08-13